# Tutorial T01a: Matrix Vectorization Operations.

The vecup module provides low-level operations for converting between
matrices and vectors. These operations follow BHATLIB's row-based
arrangement convention, which is critical for all higher-level computations.

What you will learn:
  - vecdup: extract upper-triangular elements from a symmetric matrix
  - vecndup: extract off-diagonal upper-triangular elements
  - matdupfull: reconstruct a symmetric matrix from its vector form
  - matdupdiagonefull: build a correlation matrix from off-diagonal elements
  - vecsymmetry: the pattern matrix that relates full and vectorized forms
  - nondiag: extract all non-diagonal elements

Prerequisites: None.

Expected runtime: <5 sec


In [ ]:
import os, sys
import numpy as np
np.set_printoptions(precision=4, suppress=True)
import pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent.parent / "src"))

from pybhatlib.vecup import (
    vecdup, vecndup, matdupfull, matdupdiagonefull, vecsymmetry, nondiag,
)


## Step 1: vecdup — Vectorize a Symmetric Matrix


In [ ]:
# Example from BHATLIB paper p.6
A = np.array([
    [1, 2, 3],
    [2, 4, 5],
    [3, 5, 6],
], dtype=float)

v = vecdup(A)
print(f"\n  A =\n{A}")
print(f"\n  vecdup(A) = {v}")
print(f"  Expected:    [1. 2. 3. 4. 5. 6.]")
print(f"\n  Convention: row-by-row, upper triangular including diagonal")
print(f"  Row 1: [1, 2, 3]  ->  1, 2, 3")
print(f"  Row 2:    [4, 5]  ->  4, 5")
print(f"  Row 3:       [6]  ->  6")


## Step 2: vecndup — Off-Diagonal Elements Only


In [ ]:
v_offdiag = vecndup(A)
print(f"\n  vecndup(A) = {v_offdiag}")
print(f"  Expected:    [2. 3. 5.]")
print(f"\n  This extracts just the unique off-diagonal elements.")
print(f"  For K=3: K*(K-1)/2 = {3*2//2} elements")


## Step 3: matdupfull — Reconstruct from Vector


In [ ]:
A_reconstructed = matdupfull(v)
print(f"\n  matdupfull(vecdup(A)) =\n{A_reconstructed}")
print(f"\n  Roundtrip check: {np.allclose(A, A_reconstructed)}")

# Also works from a length-6 vector directly
v2 = np.array([10., 20., 30., 40., 50., 60.])
M = matdupfull(v2)
print(f"\n  matdupfull([10,20,30,40,50,60]) =\n{M}")
print(f"  Always symmetric: M == M.T is {np.allclose(M, M.T)}")


## Step 4: matdupdiagonefull — Build Correlation Matrix


In [ ]:
# Off-diagonal correlations for 3x3 matrix
rho = np.array([0.6, 0.5, 0.5])
R = matdupdiagonefull(rho)
print(f"\n  Off-diagonal correlations: {rho}")
print(f"  matdupdiagonefull(rho) =\n{R}")
print(f"\n  Notice: diagonal is always 1 (unit diagonal)")
print(f"  This is ideal for building correlation matrices.")


## Step 5: vecsymmetry — Position Pattern Matrix


In [ ]:
S = vecsymmetry(np.zeros((3, 3)))
print(f"\n  vecsymmetry for K=3: shape = {S.shape}")
print(f"  S relates the K*(K+1)/2 = 6 unique elements to the K^2 = 9 full ones.")
print(f"  Each row of S marks BOTH symmetric positions of one upper element,")
print(f"  so S acts as the duplication pattern (not a plain selection matrix).")
print(f"\n  S =\n{S}")


Two exact identities follow from how S is built. Read carefully: S has a 1
in *both* symmetric slots for each off-diagonal element, so multiplying S by
the full row-vectorized symmetric matrix would double the off-diagonals. The
correct relationships are:

  (1) Selection from the upper triangle:    S @ vec(triu(A)) = vecdup(A)
  (2) Duplication back to the full matrix:  reshape(S.T @ vecdup(A)) = A

Identity (2) is the classic "duplication matrix" property: S.T expands the
unique elements back into the full symmetric matrix.

In [ ]:
# Identity (1): select unique elements from the upper-triangular vec
vec_triu_A = np.triu(A).T.ravel()    # row-vectorize, lower part zeroed
vecdup_A = vecdup(A)
selected = S @ vec_triu_A
print(f"  vec(triu(A))     = {vec_triu_A}")
print(f"  S @ vec(triu(A)) = {selected}")
print(f"  vecdup(A)        = {vecdup_A}")
print(f"  Identity (1) holds: {np.allclose(vecdup_A, selected)}")

# Identity (2): duplicate the unique elements back into the full matrix
A_full = (S.T @ vecdup_A).reshape(3, 3)
print(f"\n  reshape(S.T @ vecdup(A)) =\n{A_full}")
print(f"  Identity (2) holds: {np.allclose(A, A_full)}")


## Step 6: nondiag — All Non-Diagonal Elements


In [ ]:
nd = nondiag(A)
print(f"\n  A =\n{A}")
print(f"\n  nondiag(A) = {nd}")
print(f"  (row-by-row, both upper and lower triangular)")
print(f"\n  For K=3: K^2 - K = {9-3} non-diagonal elements")

# ============================================================
#  Summary
# ============================================================
print("\n" + "=" * 60)
print("  Summary: Vectorization Dimensions")
print("=" * 60)

for K in [2, 3, 4, 5]:
    n_dup = K * (K + 1) // 2
    n_ndup = K * (K - 1) // 2
    n_nondiag = K * K - K
    print(f"  K={K}: vecdup={n_dup}, vecndup={n_ndup}, nondiag={n_nondiag}")

print(f"\n  Next: t01b_ldlt.py — LDLT decomposition for positive definite matrices")
